### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install numpy
%pip install matplotlib

### 02 - Importação (Recursos)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

print(f"Numpy Version: {np.__version__}")
print(f"\nCSV Version: {csv.__version__}")

### 03 - Preparação do Dataset

In [ ]:
def load_csv_data(file_path):
    X, y = [], []

    with open(file=file_path) as file:
        reader = csv.reader(file)

        next(reader)

        for row in reader:
            X.append([int(row[0]), float(row[1])])

            y.append(int(row[2]))

    return np.array(X), np.array(y)

X, y = load_csv_data("./dataset.csv")

# Saída.

print(f"Formato dos Dados:\n{X.shape}\n{y.shape}")
print(f"\nPrimeiras Amostras:\n{X[:5]}")
print(f"\nRótulos:\n{y[:5]}")

### 04 - Função de Entropia

In [ ]:
def entropy(y):
    classes, countage = np.unique(y, return_counts=True)

    probs = countage / len(y)

    return -np.sum(probs * np.log2(probs + 1e-9))

# Teste.

print(f"Entropia ([0,0,0,0]):\n{entropy([0,0,0,0])}")
print(f"\nEntropia ([0,1,0,1]):\n{entropy([0,1,0,1])}")

### 05 - Divisão dos Dados

In [ ]:
def divide_data(X, y, feature, threshold):
    left_idx = X[:, feature] <= threshold

    right_idx = X[:, feature] > threshold

    return X[left_idx], X[right_idx], y[left_idx], y[right_idx]

# Teste.

X_left, X_right, y_left, y_right = divide_data(X, y, 0, 40)

print(f"Esquerda\n{y_left}")
print(f"\nDireita:\n{y_right}")

### 06 - Implementação da Estrutura da Árvore de Decisão

In [ ]:
class Node:
    def __init__(
        self,
        feature = None,
        threshold = None,
        left = None,
        right = None,
        value = None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class DecisionTree:
    def __init__(
        self,
        max_depth = 3,
        min_sample_quantity = 2
    ):
        self.max_depth = max_depth
        self.min_sample_quantity = min_sample_quantity
        self.root = None

    def grow(self, X, y, depth):
        sample_quantity, feature_quantity = X.shape

        class_quantity = len(np.unique(y))

        if (depth >= self.max_depth or class_quantity == 1 or sample_quantity < self.min_sample_quantity):
            final_class = np.bincount(y).argmax()

            return Node(value=final_class)
        
        best_feature, best_threshold, best_info = None, None, -1

        best_division = None

        for feature in range(feature_quantity):
            values = np.unique(X[:, feature])

            for threshold in values:
                X_left, X_right, y_left, y_right = divide_data(X, y, feature, threshold)

                if (len(y_left) == 0 or len(y_right) == 0):
                    continue

                gain = entropy(y) - (
                   (len(y_left) / sample_quantity * entropy(y_left)) + (len(y_right) / sample_quantity * entropy(y_right))
                )

                if (gain > best_info):
                    best_feature, best_threshold, best_info = feature, threshold, gain

                    best_division = (X_left, y_left, X_right, y_right)

        if (best_info == -1):
            final_class = np.bincount(y).argmax()

            return Node(value=final_class)
        
        left = self.grow(best_division[0], best_division[1], depth + 1)

        right = self.grow(best_division[2], best_division[3], depth + 1)

        return Node(best_feature, best_threshold, left, right)

    def adjust(self, X, y):
        self.root = self.grow(X, y, depth=0)

    def predict_sample(self, X, node):
        if (node.value is not None):
            return node.value
        elif (X[node.feature] <= node.threshold):
            return self.predict_sample(X, node.left)
        else:
            return self.predict_sample(X, node.right)

    def predict(self, X):
        return np.array([self.predict_sample(value, self.root) for value in X])

### 07 - Treinamento da Árvore de Decisão

In [ ]:
decision_tree = DecisionTree(max_depth=3)

decision_tree.adjust(X, y)

y_predict = decision_tree.predict(X)

accuracy = np.mean(y_predict == y)

print(f"Acurácia (Conjunto de Treino): {(accuracy * 100):.2f}%")

### 08 - Gerando o Gráfico

In [ ]:
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1

y_min, y_max = X[:, 1].min() - 500, X[:, 1].max() + 500

x_coordinates, y_coordinates = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

Z = decision_tree.predict(np.c_[x_coordinates.ravel(), y_coordinates.ravel()])

Z = Z.reshape(x_coordinates.shape)

plt.contourf(x_coordinates, y_coordinates, Z, alpha=0.4, cmap=plt.cm.Paired)

plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", cmap=plt.cm.Paired)

plt.title("Fronteira de Decisão")

plt.xlabel("Idade")
plt.ylabel("Salário")

plt.show()